<a href="https://colab.research.google.com/github/Storm00212/Data-science-and-ml-resource/blob/main/Simple_snn_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# =============================================
# Simple Spiking Neural Network (SNN) in Google Colab
# Using snnTorch + PyTorch on MNIST (rate-coded)
# =============================================

# 1. Install dependencies
!pip install snntorch torchvision matplotlib -q

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import snntorch as snn
from snntorch import spikegen, spikeplot as splt
from snntorch import functional as SF

import matplotlib.pyplot as plt
import numpy as np
import itertools

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =============================================
# 2. Load MNIST data and encode to spikes (rate coding)
# =============================================
batch_size = 128
data_path = './data/mnist'

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(data_path, train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(data_path, train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

# Spike encoding parameters
num_steps = 25  # time steps (simulation length)

# =============================================
# 3. Define a simple Feedforward SNN
# =============================================
class Net(nn.Module):
    def __init__(self, num_inputs=784, num_hidden=128, num_outputs=10, beta=0.95):
        super().__init__()

        # Layers
        self.fc1 = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.Leaky(beta=beta)

        self.fc2 = nn.Linear(num_hidden, num_outputs)
        self.lif2 = snn.Leaky(beta=beta)

    def forward(self, x):
        # x shape: (batch, num_steps, features) or we'll handle it inside

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()

        spk2_rec = []  # record output spikes
        mem2_rec = []  # record output membrane

        for step in range(num_steps):
            cur1 = self.fc1(x[step])          # input at time step
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            spk2_rec.append(spk2)
            mem2_rec.append(mem2)

        return torch.stack(spk2_rec, dim=0), torch.stack(mem2_rec, dim=0)

net = Net().to(device)

Using device: cpu


100%|██████████| 9.91M/9.91M [00:00<00:00, 145MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 51.7MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 109MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.79MB/s]
